# Lecture 5 — Exact Diagonalisation I: Hilbert Space

---

## Overview

Exact diagonalisation (ED) is the most direct approach to quantum many-body physics: represent the Hamiltonian as an explicit matrix and compute its eigenvalues and eigenvectors numerically. There is no approximation beyond floating-point arithmetic.

In this lecture we build the foundation: how to represent the many-body Hilbert space, how to encode basis states as integers, and how to construct the TFIM Hamiltonian as a sparse matrix. Lecture 06 then covers what you can extract from the resulting eigenstates.

---

In [ ]:
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import eigsh
import matplotlib.pyplot as plt
import time

plt.rcParams.update({'font.size': 12, 'figure.dpi': 100})

## 1. The Many-Body Hilbert Space

For a chain of $N$ spin-$1/2$ sites, the Hilbert space is:

$$\mathcal{H} = \underbrace{\mathbb{C}^2 \otimes \cdots \otimes \mathbb{C}^2}_{N} \cong \mathbb{C}^{2^N}$$

Its dimension is $2^N$ — exponential in system size. The **computational basis** consists of all $2^N$ product states $|s_1 s_2 \cdots s_N\rangle$ where $s_i \in \{\uparrow, \downarrow\}$.

---

In [ ]:
# Hilbert space dimension grows exponentially
print(f"{'N':>5}  {'dim = 2^N':>12}  {'memory (complex128, MB)':>25}")
print("-" * 47)
for N in [4, 8, 12, 16, 20, 24, 28, 32]:
    dim = 2**N
    mem_MB = dim * 16 / 1e6  # 16 bytes per complex128
    marker = " <- ED limit" if N == 20 else ""
    print(f"{N:>5}  {dim:>12,}  {mem_MB:>22.1f} MB{marker}")

## 2. Representing Basis States

Each computational basis state $|s_1 s_2 \cdots s_N\rangle$ is encoded as an integer from $0$ to $2^N - 1$ by treating the spin values as bits:

- $|\uparrow\rangle \to 1$, $|\downarrow\rangle \to 0$
- State $|s_1 s_2 \cdots s_N\rangle$ maps to the integer with $s_i$ in bit position $i$

Key bit operations:
- **Spin at site $i$:** `(state >> i) & 1`
- **Flip spin at site $i$:** `state ^ (1 << i)`
- **Check alignment of $i$ and $j$:** compare the two bits

---

In [ ]:
def state_to_str(state, N):
    """Convert integer state to spin string (site 0 on left)."""
    return ''.join('↑' if (state >> i) & 1 else '↓' for i in range(N))


# Show the full computational basis for N=3
N = 3
print(f"Computational basis for N={N} (dim={2**N}):")
print(f"{'Integer':>10}  {'Binary':>8}  {'Spin state':>12}")
print("-" * 35)
for state in range(2**N):
    print(f"{state:>10}  {state:>08b}  {state_to_str(state, N):>12}")

print()

# Demonstrate bit operations
state = 0b101  # |↑↓↑⟩
print(f"State: {state_to_str(state, 3)} (integer {state})")
print(f"  Spin at site 0: {'↑' if (state >> 0) & 1 else '↓'}")
print(f"  Spin at site 1: {'↑' if (state >> 1) & 1 else '↓'}")
flipped = state ^ (1 << 1)  # flip site 1
print(f"  After flipping site 1: {state_to_str(flipped, 3)} (integer {flipped})")

## 3. Building the Hamiltonian Matrix

The TFIM Hamiltonian $\hat{H} = -J \sum_{i} \hat{\sigma}^z_i \hat{\sigma}^z_{i+1} - \Gamma \sum_i \hat{\sigma}^x_i$ contributes two types of matrix elements:

- **Ising coupling** (diagonal): for each bond $(i, i+1)$, add $-J$ if spins are aligned, $+J$ if anti-aligned.
- **Transverse field** (off-diagonal): for each site $i$, add matrix element $-\Gamma$ connecting state $n$ to state $n \oplus (1 \ll i)$.

The matrix has $O(N \cdot 2^N)$ non-zero entries — very sparse.

---

In [ ]:
def build_tfim_sparse(N, J=1.0, Gamma=1.0):
    """
    Build the TFIM Hamiltonian as a CSR sparse matrix.
    Uses COO format (row, col, data) for efficient assembly.
    """
    dim = 2**N
    rows, cols, data = [], [], []

    for state in range(dim):
        diag = 0.0

        # Ising coupling: diagonal terms
        for i in range(N - 1):
            sz_i = 1 if (state >> i) & 1 else -1
            sz_j = 1 if (state >> (i + 1)) & 1 else -1
            diag -= J * sz_i * sz_j

        if diag != 0.0:
            rows.append(state)
            cols.append(state)
            data.append(diag)

        # Transverse field: off-diagonal terms
        for i in range(N):
            flipped = state ^ (1 << i)
            rows.append(state)
            cols.append(flipped)
            data.append(-Gamma)

    return sp.csr_matrix((data, (rows, cols)), shape=(dim, dim))


# Inspect sparsity
N = 8
H = build_tfim_sparse(N, J=1.0, Gamma=1.0)
dim = 2**N
nnz = H.nnz
print(f"N={N}: matrix size {dim}x{dim}, non-zeros: {nnz:,}")
print(f"Sparsity: {nnz / dim**2 * 100:.2f}% filled ({nnz} of {dim**2:,} entries)")
print(f"Non-zeros per row: {nnz / dim:.1f} (= N+1 = {N+1})")

In [ ]:
# Visualise the sparsity pattern for a small system
N_vis = 4
H_vis = build_tfim_sparse(N_vis, J=1.0, Gamma=1.0)

fig, ax = plt.subplots(figsize=(5, 5))
ax.spy(H_vis, markersize=8)
ax.set_title(f'Sparsity pattern of TFIM Hamiltonian, N={N_vis}')
labels = [state_to_str(s, N_vis) for s in range(2**N_vis)]
ax.set_xticks(range(2**N_vis))
ax.set_yticks(range(2**N_vis))
ax.set_xticklabels(labels, rotation=90, fontsize=8)
ax.set_yticklabels(labels, fontsize=8)
plt.tight_layout()
plt.show()

## 4. Diagonalisation

Two regimes:

- **Full spectrum** (`numpy.linalg.eigh`): all $2^N$ eigenvalues. Practical up to $N \sim 16$–18. Memory $O(4^N)$.
- **Ground state only** (`scipy.sparse.linalg.eigsh` with Lanczos): only the lowest few eigenvalues. Practical up to $N \sim 30$–40. Memory $O(2^N)$.

---

In [ ]:
# Compare full diagonalisation vs Lanczos for increasing N
print(f"{'N':>5}  {'Full eigh (ms)':>16}  {'Lanczos (ms)':>14}  {'E0 (full)':>12}  {'E0 (Lanczos)':>13}  {'match':>6}")
print("-" * 75)

for N in [6, 8, 10, 12, 14]:
    H = build_tfim_sparse(N, J=1.0, Gamma=1.0)

    # Full diagonalisation
    t0 = time.time()
    evals_full = np.sort(np.linalg.eigvalsh(H.toarray()))
    t_full = (time.time() - t0) * 1000

    # Lanczos
    t0 = time.time()
    evals_lanczos = eigsh(H, k=2, which='SA', return_eigenvectors=False)
    evals_lanczos = np.sort(evals_lanczos)
    t_lanczos = (time.time() - t0) * 1000

    match = np.isclose(evals_full[0], evals_lanczos[0], rtol=1e-8)
    print(f"{N:>5}  {t_full:>14.1f}ms  {t_lanczos:>12.1f}ms  {evals_full[0]:>12.6f}  {evals_lanczos[0]:>13.6f}  {'✓' if match else '✗':>6}")

## 5. Symmetries and Block Diagonalisation

The TFIM has a $\mathbb{Z}_2$ symmetry: the **parity operator** $\hat{P} = \prod_i \hat{\sigma}^x_i$ commutes with $\hat{H}$. Every eigenstate has definite parity $P = \pm 1$, and the Hamiltonian is block-diagonal in the two parity sectors, each of dimension $2^{N-1}$.

---

In [ ]:
def parity_sectors(N):
    """
    Return sorted lists of basis state indices for parity P=+1 and P=-1 sectors.
    The parity of a state is (-1)^(number of down spins), i.e., (-1)^(N - popcount).
    Equivalently: parity = (-1)^(N - bin(state).count('1')).
    """
    even, odd = [], []
    for state in range(2**N):
        n_up = bin(state).count('1')
        # Parity eigenvalue under sigma_x product: +1 if N - n_up is even
        if (N - n_up) % 2 == 0:
            even.append(state)
        else:
            odd.append(state)
    return np.array(even), np.array(odd)


def build_tfim_sector(N, J, Gamma, sector='even'):
    """Build TFIM Hamiltonian restricted to a parity sector."""
    even_idx, odd_idx = parity_sectors(N)
    idx = even_idx if sector == 'even' else odd_idx
    H_full = build_tfim_sparse(N, J, Gamma).toarray()
    return H_full[np.ix_(idx, idx)]


N = 10
even_idx, odd_idx = parity_sectors(N)
print(f"N={N}: full dim={2**N}, even sector={len(even_idx)}, odd sector={len(odd_idx)}")

# Verify: ground state is in even sector (disordered phase)
H_even = build_tfim_sector(N, J=1.0, Gamma=1.5, sector='even')
H_odd  = build_tfim_sector(N, J=1.0, Gamma=1.5, sector='odd')
E_even = np.min(np.linalg.eigvalsh(H_even))
E_odd  = np.min(np.linalg.eigvalsh(H_odd))
print(f"\nΓ/J=1.5 (disordered): E0(even)={E_even:.6f}, E0(odd)={E_odd:.6f}")
print(f"Ground state sector: {'even' if E_even < E_odd else 'odd'}")

# Ordered phase: near-degenerate ground states in both sectors
H_even_ord = build_tfim_sector(N, J=1.0, Gamma=0.3, sector='even')
H_odd_ord  = build_tfim_sector(N, J=1.0, Gamma=0.3, sector='odd')
E_even_ord = np.min(np.linalg.eigvalsh(H_even_ord))
E_odd_ord  = np.min(np.linalg.eigvalsh(H_odd_ord))
print(f"\nΓ/J=0.3 (ordered):   E0(even)={E_even_ord:.6f}, E0(odd)={E_odd_ord:.6f}")
print(f"Gap between sectors: {abs(E_even_ord - E_odd_ord):.2e} (exponentially small)")

## 6. Limitations of Exact Diagonalisation

The exponential scaling of the Hilbert space is the fundamental limitation. Memory is usually the binding constraint: storing one eigenvector requires $16 \times 2^N$ bytes (complex128).

ED is indispensable as a **benchmark**: any numerical approach must agree with ED for system sizes where both are applicable. When DMRG (lecture 09) gives an energy within $10^{-8}$ of ED, that validates the tensor network method.

---

In [ ]:
# Benchmark: wall-clock time and memory for Lanczos ED vs system size
print(f"{'N':>5}  {'dim':>10}  {'time (ms)':>12}  {'E0/N':>10}")
print("-" * 45)

for N in [8, 10, 12, 14, 16, 18, 20]:
    H = build_tfim_sparse(N, J=1.0, Gamma=1.0)
    t0 = time.time()
    evals = eigsh(H, k=1, which='SA', return_eigenvectors=False)
    elapsed = (time.time() - t0) * 1000
    print(f"{N:>5}  {2**N:>10,}  {elapsed:>10.1f}ms  {evals[0]/N:>10.6f}")

print(f"\nExact limit (N→∞): E0/N = -2/π = {-2/np.pi:.6f}")